In [26]:
# 기존 NumPy를 제거해 호환되는 버전을 새로 설치할 준비를 합니다.
!pip uninstall numpy -y

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


You can safely remove it manually.
You can safely remove it manually.


In [27]:
# PyTorch와 관련 오디오/비전 패키지를 제거해 CUDA 버전을 맞출 준비를 합니다.
!pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.4.0+cu118
Uninstalling torch-2.4.0+cu118:
  Successfully uninstalled torch-2.4.0+cu118
Found existing installation: torchvision 0.19.0+cu118
Uninstalling torchvision-0.19.0+cu118:
  Successfully uninstalled torchvision-0.19.0+cu118
Found existing installation: torchaudio 2.4.0+cu118
Uninstalling torchaudio-2.4.0+cu118:
  Successfully uninstalled torchaudio-2.4.0+cu118


You can safely remove it manually.
You can safely remove it manually.


In [28]:
# PyTorch 생태계에서 사용하는 추가 연산 패키지를 제거합니다.
!pip uninstall ultralytics-thop -y

Found existing installation: ultralytics-thop 2.1.6
Uninstalling ultralytics-thop-2.1.6:
  Successfully uninstalled ultralytics-thop-2.1.6


In [29]:
# 메모리 절약용 xformers를 기존 버전에서 제거합니다.
!pip uninstall xformers -y

Found existing installation: xformers 0.0.27.post2+cu118
Uninstalling xformers-0.0.27.post2+cu118:
  Successfully uninstalled xformers-0.0.27.post2+cu118


You can safely remove it manually.


In [30]:
# CUDA 11.8 환경에 맞는 PyTorch 2.4.0 세트를 의존성 검사 없이 설치합니다.
!pip install --ignore-installed --no-deps torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached torch-2.4.0%2Bcu118-cp311-cp311-win_amd64.whl (2692.5 MB)
  Using cached torchvision-0.19.0%2Bcu118-cp311-cp311-win_amd64.whl (5.0 MB)
  Using cached torchaudio-2.4.0%2Bcu118-cp311-cp311-win_amd64.whl (4.0 MB)

   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ------------- -------------------------- 1/3 [torchvision]
   ------------- -------------------------- 1/3 [torchvision]
   ------------- -------------------------- 1/3 [torchvision]
   ------------- -------------------------- 1/3 [torchvision]
   -------------------------- ------------- 2/3 [torch]
   -------------------------- ------------- 2/3 [torch]
   -------------------------- ------------- 2/3 [torch]
   -------------------------- ------------- 2/3 [torch]
   -------------------------- ------------- 2/3 [torch]
   -------------------------- ------------- 2/3 [torch]
   ------

In [31]:
# 다른 라이브러리와 호환되는 NumPy 버전을 설치합니다.
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.4.152 requires ultralytics-thop>=2.1.6, which is not installed.
ultralytics 8.4.152 requires torch!=2.4.0,>=1.8.0; sys_platform == "win32", but you have torch 2.4.0+cu118 which is incompatible.


In [32]:
# Windows에서 사용할 Triton 구현을 의존성 검사 없이 설치합니다.
!pip install triton-windows==3.1.0.post17 --no-deps

In [33]:
# CUDA 11.8용 xformers를 설치해 추론 시 GPU 메모리를 절약합니다.
!pip install -U xformers --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached xformers-0.0.27.post2%2Bcu118-cp311-cp311-win_amd64.whl (10.8 MB)


In [34]:
# 이미지 생성에 필요한 Diffusers, 가속 라이브러리와 .env 로더를 설치합니다.
!pip install diffusers==0.30.3 transformers==4.44.2 accelerate==0.34.2 safetensors==0.4.5 python-dotenv

In [35]:
# Stable Diffusion 파이프라인 클래스를 불러옵니다.
from diffusers import StableDiffusionPipeline
# GPU 연산과 텐서 자료형을 다루기 위해 PyTorch를 불러옵니다.
import torch
# 출력 폴더 경로를 운영체제와 무관하게 다루기 위해 Path를 불러옵니다.
from pathlib import Path
# 생성 진행률을 표시하기 위해 tqdm을 불러옵니다.
from tqdm import tqdm
# 프롬프트의 조건을 무작위로 선택하기 위해 random을 불러옵니다.
import random
# 현재 코드에서는 사용하지 않지만 파일 복사에 사용할 수 있는 모듈입니다.
import shutil

In [36]:
# 운영체제 환경변수와 .env 파일을 읽는 기능을 불러옵니다.
import os
from dotenv import load_dotenv
# 프로젝트 폴더의 .env 파일에 저장된 값을 환경변수로 불러옵니다.
load_dotenv()
# Hugging Face 토큰을 환경변수에서 읽습니다.
huggingface_token = os.environ["HUGGINGFACE_TOKEN"]

In [37]:
# 사전 학습된 DreamShaper 모델을 불러오고 반정밀도(float16)로 설정합니다.
pipe = StableDiffusionPipeline.from_pretrained(
    # 사용할 Stable Diffusion 모델의 Hugging Face 저장소입니다.
    "Lykon/dreamshaper-8",
    # GPU 메모리를 줄이기 위해 float16으로 모델을 로드합니다.
    torch_dtype=torch.float16,
    # 비공개 모델 접근에 사용할 인증 토큰입니다.
    token=huggingface_token,
    # 안전한 텐서 형식인 safetensors 가중치를 우선 사용합니다.
    use_safetensors=True
# 모델을 CUDA GPU로 이동합니다.
).to("cuda")

# xformers의 메모리 효율적인 attention을 활성화합니다.
pipe.enable_xformers_memory_efficient_attention()
# VAE가 이미지를 작은 조각으로 처리하도록 설정합니다.
pipe.enable_vae_slicing()
# 큰 이미지를 타일 단위로 처리하도록 설정합니다.
pipe.enable_vae_tiling()

Couldn't connect to the Hub: 401 Client Error. (Request ID: Root=1-6aa8eda0-3d6f5356234ed8364c752c74;34ca5c4b-87c4-4d1d-a101-5ad9c066aa20)

Repository Not Found for url: https://huggingface.co/api/models/Lykon/dreamshaper-8.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
User Access Token "mask_person_generate" is expired.
Will try to load from local cache.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [38]:
# 생성 이미지를 저장할 최상위 폴더입니다.
output_dir = Path("mask_images")
# 생성할 이미지의 가로와 세로 크기입니다.
image_size = (640, 640)
# 한 번에 GPU에 전달할 이미지 수입니다.
batch_size = 20
# 클래스별로 생성할 이미지 수입니다.
images_per_class = 1000
# 이미지 한 장을 생성할 때 반복할 확산 단계 수입니다.
num_inference_steps = 25

In [39]:
# 이미지에 등장할 사람의 성별과 인종/민족 표현 후보입니다.
genders = [
    "asian man", "asian woman", "white man", "white woman",
    "black man", "black woman", "latino man", "latino woman",
    "middle eastern man", "middle eastern woman",
    # 추가 인종/민족 후보입니다.
    "south asian man", "south asian woman",       # 인도, 파키스탄 등
    "southeast asian man", "southeast asian woman", # 베트남, 필리핀, 태국 등
    "mixed race man", "mixed race woman",
    "native american man", "native american woman",
    "pacific islander man", "pacific islander woman",
]

In [40]:
# 얼굴이나 인물의 방향과 자세 후보입니다.
poses = [
    "front view",  # 정면을 바라보는 모습
    "side view",  # 옆모습
    "three-quarter view",  # 약 45도 방향
    "looking up",  # 위를 보는 모습
    "looking down",  # 아래를 보는 모습
    "head tilted left",  # 머리를 왼쪽으로 기울인 모습
    "head tilted right"  # 머리를 오른쪽으로 기울인 모습
    # 추가 각도
    "profile view left",       # 완전 측면 (왼쪽)
    "profile view right",      # 완전 측면 (오른쪽)
    "back three-quarter view", # 뒤통수 쪽 3/4 각도 (마스크 착용시 끈 확인용)
    "chin down, eyes up",      # 턱은 내리고 눈만 위로
    "head tilted back",        # 고개 젖힘 (마스크 위치 흘러내림 케이스 포함)

    # 거리/스케일 변화
    "close-up face",
    "medium distance",
    "far distance (small face in frame)",

    # 표정/동작 변화 (마스크 상태에 영향 줄 수 있는 것들)
    "talking / mouth open",
    "eating or drinking gesture",
    "yawning",
    "sneezing or coughing gesture",

    # 마스크 착용 관련 특수 케이스
    "mask worn incorrectly (below nose)",
    "mask worn incorrectly (below chin)",
    "mask half removed",
    "hand adjusting mask",
    "hand near face (occlusion)",

    # 액세서리/조합 변수
    "wearing glasses",
    "wearing sunglasses",
    "wearing hat or cap",
    "hair covering part of face",

    # 조명/환경 변화 (포즈는 아니지만 로버스트니스에 중요)
    "backlit / strong shadow on face",
    "low light condition",
    "outdoor bright sunlight",
]

In [41]:
# 인물이 등장할 장소와 배경 후보입니다.
background = [
    "indoor", "outdoor", "office", "street", "cafe",
    "school", "subway", "park", "shopping mall",
    "restaurant", "library"
]

In [42]:
# 인물의 의상과 소품 후보입니다.
clothes = [
    "in casual clothes", "wearing a hoodie", "in a business suit",
    "in a school uniform", "wearing sportswear", "wearing traditional clothes",
    # 추가 의상과 소품 후보입니다.
    "wearing a winter coat",
    "wearing a jacket",
    "wearing a t-shirt and jeans",
    "wearing a dress",
    "wearing a raincoat",
    "wearing a lab coat / medical uniform",
    "wearing a work uniform (e.g. delivery, service)",
    "wearing a scarf",
    "wearing a beanie or knit hat",
    "wearing glasses",
    "wearing a backpack",
    "wearing a face shield in addition to mask",
    "wearing gloves",
    "wearing a vest or padding jacket",
]

In [43]:
# 마스크를 쓴 사람에게 적용할 마스크 색상 후보입니다.
mask_colors = ["white", "beige", "blue", "black", "gray", "pink"]

In [44]:
# 인물의 표정과 행동 후보입니다.
expressions = [
    "smiling",  # 미소 짓는 표정
    "neutral expression",  # 무표정
    "frowning",  # 찡그린 표정
    "surprised expression",  # 놀란 표정
    "serious expression",  # 진지한 표정
    "happy expression",  # 행복한 표정
    # 추가 표정과 행동 후보입니다.
    "laughing",
    "sad expression",
    "angry expression",
    "worried / anxious expression",
    "tired / sleepy expression",
    "confused expression",
    "yawning",
    "talking (mouth open)",
    "eyes closed",
    "winking",
    "disgusted expression",
    "crying",
]

In [45]:
# 카메라와 이미지에 담기는 인물 범위 후보입니다.
shot_types = [
    "close-up face",  # 얼굴을 가까이 촬영
    "portrait, head and shoulders",  # 머리와 어깨 중심
    "upper body portrait",  # 상반신
    "full body portrait",  # 전신
    # 추가 촬영 구도 후보입니다.
    "extreme close-up (eyes and nose area)",
    "profile close-up",
    "waist-up shot",
    "three-quarter body shot",
    "wide shot (person small in frame, background visible)",
    "over-the-shoulder shot",
    "low angle shot",
    "high angle shot",
    "candid / walking shot",
]

In [46]:
# 이미지에 등장할 사람의 연령대 후보입니다.
age_groups = [
    "child (5-10 years old)",
    "teenager",
    "young adult (20s)",
    "middle-aged adult (40s-50s)",
    "elderly (65+ years old)",
]

In [47]:
# 클래스별 이미지 설명을 만들기 위한 프롬프트 형식입니다.
class_prompt_templates = {
    # 마스크를 쓴 사람을 설명하는 템플릿입니다.
    "mask_on": (
        "a {shot} of a {genders} wearing a {mask_colors} face mask covering mouth and nose properly,"
        "{expressions}, {poses}, {background}, {clothes}, {age_groups} realistic, high-quality"
    ),
    # 마스크를 쓰지 않은 사람을 설명하는 템플릿입니다.
    "no_mask": (
        "a {shot} of a {genders} without mask, {expressions}, {poses},"
        "{background}, {clothes}, {age_groups}, realistic, high-quality"
    )
}

In [48]:
# 이미지에 원하지 않는 흐림, 왜곡, 저품질 요소를 지정합니다.
negative_prompts = {
    "mask_on": "blurry, deformed face, low quality, unnatural skin",
    "no_mask": "blurry, deformed face, low quality, unnatural skin"
}

In [ ]:
# mask_on과 no_mask 두 클래스의 이미지를 차례대로 생성합니다.
for class_name, template in class_prompt_templates.items():
    # 클래스별 저장 폴더를 만들고 이미 존재해도 오류가 나지 않게 합니다.
    class_folder = output_dir / class_name
    class_folder.mkdir(parents=True, exist_ok=True)

    # 전체 이미지 수를 batch_size만큼 나누어 여러 번 생성합니다.
    for i in tqdm(range(0, images_per_class, batch_size), desc=class_name):
        # 현재 배치의 프롬프트와 부정 프롬프트 목록을 준비합니다.
        prompts, negs = [], [negative_prompts[class_name]] * batch_size

        # 배치에 포함할 이미지마다 조건을 무작위로 선택합니다.
        for _ in range(batch_size):
            prompt = template.format(
                shot=random.choice(shot_types),  # 촬영 구도를 선택합니다.
                genders=random.choice(genders),  # 성별과 인종/민족 표현을 선택합니다.
                poses=random.choice(poses),  # 자세를 선택합니다.
                background=random.choice(background),  # 배경을 선택합니다.
                clothes=random.choice(clothes),  # 의상과 소품을 선택합니다.
                # no_mask 클래스에서는 마스크 색상을 빈 문자열로 둡니다.
                mask_colors=random.choice(mask_colors) if class_name != "no_mask" else "",
                expressions=random.choice(expressions),  # 표정이나 행동을 선택합니다.
                age_groups=random.choice(age_groups)  # 연령대를 선택합니다.
            )
            # 완성된 프롬프트를 현재 배치 목록에 추가합니다.
            prompts.append(prompt)

        # 학습이 아닌 추론 모드와 CUDA 자동 자료형 변환을 사용합니다.
        with torch.inference_mode(), torch.autocast("cuda"):
            images = pipe(
                prompts,
                negative_prompt=negs,
                height=image_size[1],
                width=image_size[0],
                num_inference_steps=num_inference_steps
            ).images

        # 생성된 이미지를 클래스 폴더에 순번을 붙여 저장합니다.
        for idx, img in enumerate(images):
            img.save(class_folder / f"{class_name}_{i + idx:05}.png")

mask_on:   0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

mask_on:   2%|▏         | 1/50 [01:07<55:16, 67.68s/it]

  0%|          | 0/25 [00:00<?, ?it/s]

mask_on:   4%|▍         | 2/50 [02:18<55:42, 69.64s/it]

...

  0%|          | 0/25 [00:00<?, ?it/s]

mask_on:  38%|███▊      | 19/50 [23:09<38:05, 73.71s/it]

  0%|          | 0/25 [00:00<?, ?it/s]

mask_on:  40%|████      | 20/50 [24:23<36:46, 73.56s/it]

  0%|          | 0/25 [00:00<?, ?it/s]